# 实验：vLLM 多轮 Deep Research Agent

> 这份 notebook 对应多轮 agent 版本，用于在云端顺序执行：环境准备 -> 多轮推理 -> 生成 `submission.jsonl` -> 自动评测。

> 它不覆盖单步 baseline，而是调用 `agent/multistep_agent.py` 中的新实现。

## 0. 环境与服务说明

假设你已经在服务器上启动了 vLLM OpenAI-compatible 服务，并且当前模型支持工具调用。

这份 notebook 会：

1. 连接现有 vLLM 服务
2. 使用 `search + get_document` 跑多轮 agent
3. 生成统一格式的 `submission.jsonl`
4. 调用 `agent.eval` 自动评测

In [ ]:
!python --version
!pip install -r agent/requirements.txt

## 1. vLLM 服务配置

这里先填好模型名和服务地址。

如果使用 Qwen 工具调用服务，通常应类似：

`vllm serve ./Qwen3-8B --served-model-name qwen_auto --enable-auto-tool-choice --tool-call-parser hermes ...`

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
MODEL_NAME = 'qwen_auto'
# MODEL_NAME = 'pangu_auto'
API_KEY = 'dummy'

## 2. 服务器上预构建 BM25 索引

正式运行使用 `browsecomp-plus-corpus` 全量语料。

索引一般只构建一次；如果已经建好，可以直接跳过执行。

In [ ]:
corpus_path = str('browsecomp-plus-corpus')
bm25_index_path = str('indexes/browsecomp_plus_bm25.sqlite')

# 首次在服务器上执行；已经建过就跳过
!python -m agent.build_bm25_index --corpus-path ./browsecomp-plus-corpus --index-path ./indexes/browsecomp_plus_bm25.sqlite --overwrite

## 3. 初始化多轮 agent 依赖

In [ ]:
from agent.vllm_client import VLLMClient
from agent.tools import build_searcher, get_agent_tool_specs_and_registry
from agent.dataset_utils import load_jsonl
from agent.multistep_agent import run_multistep_agent, generate_submission

hard50_path = str(project_root / 'browsecomp_plus_hard50.jsonl')
submission_path = str(project_root / 'runs' / 'multistep_submission.jsonl')
eval_output_path = str(project_root / 'runs' / 'multistep_eval_results.jsonl')

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=bm25_index_path)
tool_specs, tool_registry = get_agent_tool_specs_and_registry(searcher=searcher, k=5, snippet_max_chars=1200)
print('search_type:', searcher.search_type)

## 4. 多轮 agent 参数

建议：

- `max_rounds=7`：已经决定统一为 7
- `decision_max_tokens=512`：每轮动作决策通常够用
- `answer_max_tokens=768`：最终回答保留更宽松的预算

In [ ]:
TOP_K = 5
MAX_ROUNDS = 7
DECISION_MAX_TOKENS = 512
ANSWER_MAX_TOKENS = 768
SEARCH_SNIPPET_MAX_CHARS = 1200
TOOL_CONTENT_MAX_CHARS = 4000

## 5. 单条样本演示

先只跑一条，看 agent 是否真的出现多轮工具调用。

In [ ]:
rows = load_jsonl(hard50_path, limit=1)
demo_row = rows[0]

demo = run_multistep_agent(
    question=demo_row['query'],
    client=client,
    model_name=MODEL_NAME,
    tool_specs=tool_specs,
    tool_registry=tool_registry,
    max_rounds=MAX_ROUNDS,
    decision_max_tokens=DECISION_MAX_TOKENS,
    answer_max_tokens=ANSWER_MAX_TOKENS,
    tool_content_max_chars=TOOL_CONTENT_MAX_CHARS,
)

print('query_id:', demo_row['query_id'])
print('gold_answer:', demo_row['answer'])
print('predicted_answer:', demo['predicted_answer'])
print('messages:', len(demo['messages']))
print('\nmessage roles:')
for msg in demo['messages']:
    print('-', msg['role'], list(msg.keys()))

## 6. 批量生成 submission.jsonl

这里会按统一格式写出多轮 agent 的轨迹文件。

In [ ]:
rows = load_jsonl(hard50_path, limit=50)

records = generate_submission(
    dataset_rows=rows,
    index_path=bm25_index_path,
    base_url=VLLM_BASE_URL,
    model_name=MODEL_NAME,
    output_path=submission_path,
    api_key=API_KEY,
    top_k=TOP_K,
    search_snippet_max_chars=SEARCH_SNIPPET_MAX_CHARS,
    tool_content_max_chars=TOOL_CONTENT_MAX_CHARS,
    max_rounds=MAX_ROUNDS,
    decision_max_tokens=DECISION_MAX_TOKENS,
    answer_max_tokens=ANSWER_MAX_TOKENS,
)

print('\nSaved to:', submission_path)
print('num_records:', len(records))

sample = records[0]
print('\n--- 第一条轨迹样例 ---')
print('query_id:', sample['query_id'])
print('status:', sample['status'])
print('predicted_answer:', sample['predicted_answer'])
print('messages:', len(sample['messages']))

## 7. 自动评估

注意：这里仍然依赖同一个 vLLM 服务，因为 `agent.eval` 会再次调用模型判断预测答案与标准答案是否一致。

In [ ]:
from agent.eval import run_evaluation

summary, details = run_evaluation(
    submission_path=submission_path,
    dataset_path=hard50_path,
    model_name=MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key=API_KEY,
    output_path=eval_output_path,
    temperature=0.0,
    max_tokens=256,
    verbose=True,
)

print('\nSummary:')
print(summary)

## 8. 查看评测结果样例

In [ ]:
import json
from pathlib import Path

eval_lines = [json.loads(line) for line in Path(eval_output_path).read_text(encoding='utf-8').splitlines() if line.strip()]
print('summary line:')
print(eval_lines[0])
print('\nfirst detail line:')
print(json.dumps(eval_lines[1], ensure_ascii=False, indent=2)[:3000])